# SETU — SeqKD vs DPO-distill comparison (the paper's head-to-head)

Trains the **same** student two ways at matched size and evaluates both on the
**same real-reference dev set**:

- **S1 SeqKD** — SFT on the teacher's 1-best translations (Kim & Rush 2016 baseline).
- **S2 DPO-distill (ours)** — SFT on human references + DPO on ChrF-ranked preferences.

Set **GPU T4 x2** and **Internet On**. Uses `--limit 100000` (~5-6 h total for the
distill + two trainings); lower it if your GPU budget is tighter.

In [ ]:
import torch; print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!rm -rf /kaggle/working/SETU_v2
!git clone https://github.com/GeekyRiolu/SETU_v2.git /kaggle/working/SETU_v2
%cd /kaggle/working/SETU_v2/SETU
!pip -q install -e ".[data,teacher,prefs,quantize]"
!cp configs/model.gpu.yaml configs/model.yaml
!cp configs/training.gpu.yaml configs/training.yaml
!sed -i 's/device: cpu/device: cuda/' configs/teacher.yaml

In [ ]:
LIMIT = 100000
# data + preferences (preferences only needed for S2's DPO stage)
!setu-data --limit {LIMIT + 2000}
!setu-prefs --max-entries 8000

In [ ]:
# generate the teacher-distilled corpus (teacher 1-best targets) for SeqKD.
# This is the expensive step (teacher inference over LIMIT sources).
!setu-distill --limit {LIMIT}

In [ ]:
# S1 SeqKD: SFT on teacher targets, no DPO. Eval is on real references.
!python scripts/train_full.py --train-corpus distilled --skip-dpo --limit {LIMIT} --dev-size 500
!cp checkpoints/hin_Deva-eng_Latn/train_report.json /kaggle/working/report_S1_seqkd.json

In [ ]:
# S0 SFT (human refs) + S2 SFT+DPO. Same size, same dev set.
!python scripts/train_full.py --train-corpus processed --limit {LIMIT} --dev-size 500
!cp checkpoints/hin_Deva-eng_Latn/train_report.json /kaggle/working/report_S2_dpo.json

In [ ]:
# comparison table
import json
s1 = json.load(open('/kaggle/working/report_S1_seqkd.json'))
s2 = json.load(open('/kaggle/working/report_S2_dpo.json'))
tb = s2['sft_eval'].get('teacher_bleu')
def row(name, ev):
    r = ev.get('bleu_ratio')
    print(f"{name:24s} BLEU {ev['bleu']:6.2f}  chrF {ev['chrf']:6.2f}  ratio {r if r is None else round(r,3)}")
print(f"teacher dev BLEU = {tb}\n")
row('S1 SeqKD (SFT-teacher)', s1['sft_eval'])
row('S0 SFT (human refs)',    s2['sft_eval'])
if 'dpo_eval' in s2: row('S2 SFT-ref + DPO (ours)', s2['dpo_eval'])
print('\nPaste these into docs/PAPER_PLAN.md Table 1.')